# NB 03: Cost Modeling

Transaction cost analysis: trading fees, slippage, gas costs, break-even spreads,
and sensitivity analysis across trade sizes and fee tiers.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

from src.config import EXCHANGES, FEES, FIGURES_DIR
from src.models.cost_model import (
    trading_fee_round_trip,
    total_cost_round_trip,
    break_even_spread,
    cost_sensitivity_analysis,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_style('whitegrid')

## 1. Fee Schedule Overview

In [ ]:
fee_table = pd.DataFrame(FEES).T
fee_table.columns = ['Maker', 'Taker']
fee_table['Maker (bps)'] = fee_table['Maker'] * 10_000
fee_table['Taker (bps)'] = fee_table['Taker'] * 10_000
fee_table['Type'] = [EXCHANGES[ex]['type'] for ex in fee_table.index]
print('Exchange Fee Schedule:')
fee_table

## 2. Round-Trip Costs by Exchange Pair

In [ ]:
exchanges = list(FEES.keys())
cost_records = []

for ex_a, ex_b in combinations(exchanges, 2):
    for size in [1_000, 10_000, 50_000, 100_000]:
        costs = total_cost_round_trip(ex_a, ex_b, notional_usd=size)
        costs['exchange_a'] = ex_a
        costs['exchange_b'] = ex_b
        costs['trade_size'] = size
        costs['pair_type'] = (
            'CEX-CEX' if EXCHANGES[ex_a]['type'] == 'CEX' and EXCHANGES[ex_b]['type'] == 'CEX' else
            'DEX-DEX' if EXCHANGES[ex_a]['type'] == 'DEX' and EXCHANGES[ex_b]['type'] == 'DEX' else
            'CEX-DEX'
        )
        cost_records.append(costs)

cost_df = pd.DataFrame(cost_records)

# Show for $10k notional
pivot = cost_df[cost_df['trade_size'] == 10_000].pivot_table(
    index=['exchange_a', 'exchange_b'],
    values=['fees', 'slippage', 'gas', 'total'],
    aggfunc='first'
) * 10_000  # Convert to bps

print('Round-trip costs (bps) for $10K notional:')
pivot.round(1)

## 3. Break-Even Spread Analysis

In [ ]:
# Break-even spread for each pair at various holding periods
be_records = []

for ex_a, ex_b in combinations(exchanges, 2):
    for hp in [1, 3, 7, 14, 30]:
        be = break_even_spread(ex_a, ex_b, holding_periods=hp, notional_usd=10_000)
        be_records.append({
            'exchange_a': ex_a, 'exchange_b': ex_b,
            'holding_periods': hp,
            'break_even_bps': be * 10_000,
            'pair': f'{ex_a}/{ex_b}',
        })

be_df = pd.DataFrame(be_records)

fig, ax = plt.subplots(figsize=(12, 6))
for pair in be_df['pair'].unique():
    pair_data = be_df[be_df['pair'] == pair]
    ax.plot(pair_data['holding_periods'], pair_data['break_even_bps'],
            marker='o', label=pair, alpha=0.7)

ax.set_xlabel('Holding Period (8h periods)')
ax.set_ylabel('Break-Even Spread (bps per period)')
ax.set_title('Break-Even Spread vs Holding Period')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'break_even_spreads.pdf', bbox_inches='tight')
plt.show()

## 4. Cost Sensitivity: Trade Size

In [ ]:
# Waterfall chart: cost components for a representative pair
rep_pair = ('okx', 'hyperliquid')  # CEX-DEX pair
sizes = [1_000, 5_000, 10_000, 25_000, 50_000, 100_000]

waterfall = []
for size in sizes:
    costs = total_cost_round_trip(*rep_pair, notional_usd=size)
    waterfall.append({
        'trade_size': f'${size:,.0f}',
        'Fees': costs['fees'] * 10_000,
        'Slippage': costs['slippage'] * 10_000,
        'Gas': costs['gas'] * 10_000,
    })

wf_df = pd.DataFrame(waterfall).set_index('trade_size')

fig, ax = plt.subplots(figsize=(10, 5))
wf_df.plot(kind='bar', stacked=True, ax=ax)
ax.set_title(f'Cost Breakdown: {rep_pair[0].upper()}/{rep_pair[1]} by Trade Size')
ax.set_ylabel('Cost (bps)')
ax.set_xlabel('Trade Size')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cost_waterfall.pdf', bbox_inches='tight')
plt.show()

## 5. Full Sensitivity Grid

In [ ]:
# Run sensitivity for key pairs
key_pairs = [('okx', 'hyperliquid'), ('okx', 'dydx'), ('bitmex', 'hyperliquid')]

for ex_a, ex_b in key_pairs:
    sens = cost_sensitivity_analysis(ex_a, ex_b)
    
    # Heatmap: break-even by trade size vs holding period (at 1x fees)
    pivot = sens[sens['fee_multiplier'] == 1.0].pivot_table(
        index='trade_size_usd',
        columns='holding_periods',
        values='break_even_per_period',
    ) * 10_000  # bps
    
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd_r', ax=ax)
    ax.set_title(f'Break-Even (bps/period): {ex_a}/{ex_b}')
    ax.set_ylabel('Trade Size (USD)')
    ax.set_xlabel('Holding Periods')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'sensitivity_{ex_a}_{ex_b}.pdf', bbox_inches='tight')
    plt.show()